Copyright 2021 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required bsteps_per_epoch y applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.steps_per_epoch 
# limitations under the License.

#Assess privacy risks with TensorFlow Privacy Secret Sharer Attack

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/privacy/blob/master/tensorflow_privacy/privacy/privacy_tests/secret_sharer/secret_sharer_example.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/privacy/blob/master/tensorflow_privacy/privacy/privacy_tests/secret_sharer/secret_sharer_example.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

In this colab, we demonstrate how [secret sharer](https://arxiv.org/abs/1802.08232) works on a language model. We will train a model with "secrets", i.e. random sequences, inserted in the training data, and then evaluate if the model has "memorized" those secrets.

(The code is largely based on a Tensorflow tutorial for training a character-level LSTM on Shakespeare dataset.)

#Setup

You may set the runtime to use a GPU by Runtime > Change runtime type > Hardware accelerator.

In [ ]:

%pip install tf-models-official
#!pip install git+https://github.com/tensorflow/privacy.git
#clear_output()

In [ ]:
import functools
import os
import numpy as np
import tensorflow as tf


from tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets import SecretConfig, TextSecretProperties, generate_text_secrets_and_references, construct_secret
from tensorflow_privacy.privacy.privacy_tests.secret_sharer.exposures import compute_exposure_interpolation, compute_exposure_extrapolation

# Define functions

In [ ]:


SEQ_LENGTH = 1024

BATCHES_PER_EPOCH = 200
TRAIN_EPOCHS = 50
BATCH_SIZE = 16
DATASET_SIZE = 20000




In [ ]:
import pandas as pd
def load_text_from_csv(path_to_file):
    """Carga el texto de la columna 'TEXT' de un archivo CSV."""
    df = pd.read_csv(path_to_file, usecols=["TEXT"], dtype=str, encoding="ISO-8859-1") 
    text = " ".join(df["TEXT"].dropna().iloc[:DATASET_SIZE].tolist()) 
    return text

In [ ]:
#@title Function for getting dataset combined with secrets.
def get_dataset(secrets=None,
                batch_size=None, seq_length=SEQ_LENGTH):
  """Creates a dataset from a given text file.

  Args:
    secrets: Secrets to be inserted.
    batch_size: Batch size to use.
    seq_length: The length of the LSTM sequence.

  Returns:
    A tuple, consisting of the Dataset and the class to character mapping
    and character to class mapping.
  """
  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  text = load_text_from_csv("MIMIC_NoteEvents_deanonymized.csv")

  # Create vocab
  vocab = sorted(set(text))
  char2idx = {u: i for i, u in enumerate(vocab)}
  idx2char = np.array(vocab)

  # Gather all secrets
  if not secrets:
    pass
  else:
    all_secrets = []
    for secret in secrets:
      secret.datasets = {}
      for r, seqs in secret.secrets.items():
        all_secrets += seqs * r
        indexed_seqs = np.array([tokenizer.encode(s, padding='max_length', max_length=seq_length) for s in seqs])
        secret.datasets[r] = {
            'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
            'label': list(map(lambda chunk: chunk[1:], indexed_seqs))
        }
      indexed_seqs = np.array([tokenizer.encode(s,padding='max_length', max_length=seq_length) for s in secret.references])
      secret.datasets[0] = {
          'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
          'label': list(map(lambda chunk: chunk[1:], indexed_seqs))
      }
    # Insert the secrets into the training data
    idx = np.random.choice(len(text) - 1, len(all_secrets) - 1, replace=False) + 1
    # split text into len(all_secrets) parts
    text = np.split(list(text), np.sort(idx))
    text = [''.join(t) for t in text]
    text = ''.join(np.vstack([text, all_secrets]).transpose().reshape(-1))

  # Split text into sequence length + 1 chucks to create examples
  def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text


  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.encode(text, return_tensors="tf")[0]
  seq_length = 1024  # GPT-2 usa una secuencia máxima de 1024 tokens
  dataset = tf.data.Dataset.from_tensor_slices(input_ids)
  dataset = dataset.batch(seq_length + 1, drop_remainder=True)
  def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

  dataset = dataset.map(split_input_target)
  dataset = dataset.shuffle(10000).batch(8, drop_remainder=True)  # Batch final
    
  #text_as_int = np.array([char2idx[c] for c in text])
  #char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
  #sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

  #dataset = sequences.map(split_input_target)
  #dataset = dataset.shuffle(10000).repeat()
  #dataset = dataset.batch(batch_size, drop_remainder=True)

  return dataset, idx2char, char2idx

In [37]:
from tensorflow_privacy.privacy.analysis import compute_noise_from_budget

epsilon = 10
delta = 1 / DATASET_SIZE
l2_norm_clip = 1.0 
# Calculamos el noise_multiplier para garantizar (ε, δ)-DP
noise_multiplier = compute_noise_from_budget(
    n=DATASET_SIZE,  # Tamaño del dataset
    batch_size=batch_size,
    epsilon=epsilon,
    epochs=train_epochs,
    delta=delta,
    min_noise=1e-6
)

DuplicateFlagError: The flag 'N' is defined twice. First from tensorflow_privacy.privacy.analysis.compute_dp_sgd_privacy, Second from tensorflow_privacy.privacy.analysis.compute_noise_from_budget.  Description from first occurrence: Total number of examples in the training data.

In [44]:
import os
import tensorflow as tf
from transformers import TFAutoModelForCausalLM, create_optimizer, AutoTokenizer, AdamWeightDecay
from tensorflow.keras.optimizers import SGD
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer

def train_model(dataset, batch_size, vocab_size, train_epochs, checkpoint_dir, dp):
    """Entrena un modelo Transformer con checkpoints."""

    train_steps = BATCHES_PER_EPOCH // batch_size 

    model = TFAutoModelForCausalLM.from_pretrained("gpt2")
    noise_multiplier = 0.4272711315437153
    l2_norm_clip = 1e-6
    num_microbatches = BATCH_SIZE
    if dp:
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("🔒 Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = SGD(learning_rate=0.03)

    model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), jit_compile=True)


    callbacks = []

    # Callback para guardar checkpoints
    if checkpoint_dir:
        checkpoint_filepath = os.path.join(checkpoint_dir, 'ckpt_{epoch}')
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_filepath,
            save_weights_only=True,
            save_best_only=True,  # Guarda solo el mejor modelo
            monitor="loss",  # Basado en la pérdida
            mode="min",
            verbose=1
        )
        callbacks.append(checkpoint_callback)

        # Restaurar último checkpoint si existe
        checkpoint_latest = tf.train.latest_checkpoint(checkpoint_dir)
        if checkpoint_latest:
            model.load_weights(checkpoint_latest)
            initial_epoch = int(os.path.basename(checkpoint_latest).split('_')[-1])
            print(f'✅ Checkpoint encontrado: {checkpoint_latest}. Continuando desde la época {initial_epoch}.')
        else:
            initial_epoch = 0
    else:
        initial_epoch = 0

    # Callback para medir el tiempo de cada época (opcional)
    time_callback = tf.keras.callbacks.TensorBoard(log_dir="./logs", histogram_freq=1)
    callbacks.append(time_callback)

    # Entrenar el modelo
    history = model.fit(
        dataset,
        epochs=train_epochs,
        initial_epoch=initial_epoch,
        steps_per_epoch=train_steps,
        callbacks=callbacks,
        verbose=2
    )

    return history, callbacks

In [50]:
from transformers import TFAutoModelForCausalLM
import tensorflow as tf

def load_gpt2_model(model_checkpoint, checkpoint_dir, dp):
    """Carga un modelo GPT-2 con pesos desde el último checkpoint."""
    
    # Construir el modelo GPT-2
    prediction_model = TFAutoModelForCausalLM.from_pretrained(model_checkpoint)

    
    if dp:
        noise_multiplier = 0.4272711315437153
        l2_norm_clip = 1e-6
        num_microbatches = BATCH_SIZE
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("🔒 Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = SGD(learning_rate=0.03)
    
    # Cargar los pesos del último checkpoint si existe
    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint:
        prediction_model.load_weights(latest_checkpoint)
        print(f'✅ Pesos cargados desde {latest_checkpoint}')
    else:
        print('⚠️ No se encontró ningún checkpoint, cargando pesos preentrenados.')

    # Compilar el modelo con la pérdida adecuada para modelos de lenguaje
    prediction_model.compile(optimizer = optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

    return prediction_model


In [48]:
from tensorflow.nn import softmax
#@title Function to compute perplexity.
def compute_perplexity_for_secret(secrets, checkpoint_dir, vocab_size, dp=False):
  """Get perplexity for given examples.

  Args:
    secrets: a list, where each element is a secret dictionary
    checkpoint_dir: the directory from which to load checkpoints
    vocab_size: vocabulary size

  Returns:
    Perplexity for the input examples.
  """
  prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)

  cce = tf.keras.losses.SparseCategoricalCrossentropy(reduction=tf.keras.losses.Reduction.NONE)
  ppls = [{} for _ in range(len(secrets))]
  ppls_reference = [None] * len(secrets)
  for i, secret in enumerate(secrets):
    print("i:", i)
    for r in secret.datasets.keys():
      d = secret.datasets[r]
      predictions = prediction_model.predict(d['data']).logits  
      predictions_prob = softmax(predictions, axis=-1)
      loss = np.mean(cce(d['label'], predictions_prob),
                     axis=1)
      loss = list(loss)
      if r == 0:
        print("LOSS:", loss)
        ppls_reference[i] = loss
      else:
        ppls[i][r] = loss
  return ppls, ppls_reference

# Run training and secret sharer on Shakespeare dataset.

In [35]:
#@title Generate secrets
# Here we present just one example and you can try other combinations of patterns,
# vocabs and number of repetitions. For example, you may use other patterns and
# different vocabularies for them.
patterns = ['{}'*5,
            'Your doctor is ' + '{}'*5,
            '{}{}-{}{}-{}',
            '{}' * 10]
vocabs = [
    ["Abel", "Adam", "Carl", "Dean", "Gill", "Grey", "Hunt", "Leon", "Lynn", "Reed"]
] * 4
num_repetitions = [1, 10, 100]
num_secrets_for_repetitions = [30] * len(num_repetitions)
num_references = 25 #65536
secret_configs = [SecretConfig(num_repetitions, num_secrets_for_repetitions, num_references,
                               properties=TextSecretProperties(vocab, pattern))
                  for vocab, pattern in zip(vocabs, patterns)]
secrets = generate_text_secrets_and_references(secret_configs)

# Let's look at the variable "secrets"
print(f'"secrets" is a list and the length is {len(secrets)} because we have four sets of secrets.')
print(f'The type of the element in "secrets" is {type(secrets[0])}.')
print(f'The field "config" contains its configuration: {secrets[0].config}')
print(f'The filed "secrets" is a dictionary, with key being the number of repetitions and value being a list of secrets.',
      f'For example, the secrets that are going to be repeated for 100 times are {secrets[0].secrets[100]}.')
print(f'The field "references" is a list of reference sequences that are not going to be injected. The first 10 elements are {secrets[0].references[:10]}.')

"secrets" is a list and the length is 4 because we have four sets of secrets.
The type of the element in "secrets" is <class 'tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets.SecretsSet'>.
The field "config" contains its configuration: SecretConfig(num_repetitions=[1, 10, 100], num_secrets_for_repetitions=[30, 30, 30], num_references=25, name='', properties=TextSecretProperties(vocab=['Abel', 'Adam', 'Carl', 'Dean', 'Gill', 'Grey', 'Hunt', 'Leon', 'Lynn', 'Reed'], pattern='{}{}{}{}{}'))
The filed "secrets" is a dictionary, with key being the number of repetitions and value being a list of secrets. For example, the secrets that are going to be repeated for 100 times are ['AbelDeanGreyReedGill', 'HuntHuntLynnLeonGrey', 'LeonLynnLynnLeonAbel', 'CarlDeanLynnAdamDean', 'CarlGreyGillAdamGrey', 'CarlLynnGillDeanAbel', 'GillDeanHuntReedLynn', 'CarlDeanAdamReedGrey', 'GreyReedDeanAbelGrey', 'LeonGreyGreyAbelAdam', 'ReedReedCarlAbelReed', 'LeonHuntReedGillAdam', 'HuntDeanH

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
for i in range(len(secrets)):
  dataset, idx2char, char2idx = get_dataset(
      [secrets[i]],
      batch_size=BATCH_SIZE)
    
  checkpoint_dir = 'DP-checkpoints'+str(i)
  print("dataset", i, "loaded")
  history, callbacks = train_model(dataset, batch_size=32, vocab_size=tokenizer.vocab_size, train_epochs=5, checkpoint_dir=checkpoint_dir, dp=True)

In [ ]:
#@title Secret sharer evaluation
ppls_all, ppls_reference_all = [], []
for i in range(len(secrets)):
  checkpoint_dir =  'checkpoints'+str(i)
  ppls, ppls_reference = compute_perplexity_for_secret(
      [secrets[i]], checkpoint_dir, tokenizer.vocab_size)
  ppls_all += ppls
  ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  print("p", p, "p_r",p_r)
  exposure_interpolation = compute_exposure_interpolation(p, p_r)

  exposure_extrapolation = compute_exposure_extrapolation(p, p_r)

  # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
  # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
  # the average exposure (with standard deviation).
  print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
  print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
  print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))

In [51]:
#@title Comparing exposure between base model, DP-SGD and SGD

#SGD model:
ppls_all, ppls_reference_all = [], []
checkpoint_dir =  'checkpoints0'
ppls, ppls_reference = compute_perplexity_for_secret(
  [secrets[0]], checkpoint_dir, tokenizer.vocab_size)
ppls_all += ppls
ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  if (i != 0):
      break
  else:
      exposure_interpolation = compute_exposure_interpolation(p, p_r)
    
      exposure_extrapolation = compute_exposure_extrapolation(p, p_r)
    
      # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
      # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
      # the average exposure (with standard deviation).
      print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
      print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
      print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))

#DP-SGD model:
ppls_all, ppls_reference_all = [], []
checkpoint_dir =  'DP-checkpoints0'
ppls, ppls_reference = compute_perplexity_for_secret(
  [secrets[0]], checkpoint_dir, tokenizer.vocab_size, dp=True)
ppls_all += ppls
ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  if (i != 0):
      break
  else:
      exposure_interpolation = compute_exposure_interpolation(p, p_r)
    
      exposure_extrapolation = compute_exposure_extrapolation(p, p_r)
    
      # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
      # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
      # the average exposure (with standard deviation).
      print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
      print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
      print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))
#base model (GPT2 without finetuning):
ppls_all, ppls_reference_all = [], []
checkpoint_dir =  'base'
ppls, ppls_reference = compute_perplexity_for_secret(
  [secrets[0]], checkpoint_dir, tokenizer.vocab_size)
ppls_all += ppls
ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  if (i != 0):
      break
  else:
      exposure_interpolation = compute_exposure_interpolation(p, p_r)
    
      exposure_extrapolation = compute_exposure_extrapolation(p, p_r)
    
      # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
      # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
      # the average exposure (with standard deviation).
      print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
      print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
      print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


✅ Pesos cargados desde checkpoints0/ckpt_5
i: 0
25/25 [==============================] - 14s 552ms/step
LOSS: [6.88613, 6.8852158, 6.891287, 6.8880134, 6.888434, 6.8891354, 6.8922915, 6.888119, 6.886187, 6.8844056, 6.884048, 6.8883405, 6.8847036, 6.8871126, 6.8866167, 6.885937, 6.887424, 6.884147, 6.8845387, 6.8862433, 6.886798, 6.886001, 6.887499, 6.8864756, 6.8842053]
Secrets #1, vocab=['Abel', 'Adam', 'Carl', 'Dean', 'Gill', 'Grey', 'Hunt', 'Leon', 'Lynn', 'Reed'], pattern={}{}{}{}{}
  Interpolation: repetition=1, avg_exposure=0.94±0.85; repetition=10, avg_exposure=1.37±1.58; repetition=100, avg_exposure=1.29±1.34
  Extrapolation: repetition=1, avg_exposure=1.13±1.14; repetition=10, avg_exposure=inf±nan; repetition=100, avg_exposure=inf±nan


All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


🔒 Usando DP-SGD con noise_multiplier = 0.4272711315437153
✅ Pesos cargados desde DP-checkpoints0/ckpt_2
i: 0
25/25 [==============================] - 14s 570ms/step
LOSS: [11.377692, 11.428122, 11.685152, 11.428332, 11.494751, 11.608932, 11.681629, 11.66444, 11.455784, 11.701044, 11.655578, 11.5893955, 11.501511, 11.425659, 11.780804, 11.438771, 11.656197, 11.471624, 11.444522, 11.39961, 11.420436, 11.687338, 11.483913, 11.585291, 11.515182]
Secrets #1, vocab=['Abel', 'Adam', 'Carl', 'Dean', 'Gill', 'Grey', 'Hunt', 'Leon', 'Lynn', 'Reed'], pattern={}{}{}{}{}
  Interpolation: repetition=1, avg_exposure=1.40±1.38; repetition=10, avg_exposure=1.05±1.22; repetition=100, avg_exposure=0.99±1.25
  Extrapolation: repetition=1, avg_exposure=1.79±1.94; repetition=10, avg_exposure=1.33±1.69; repetition=100, avg_exposure=1.57±3.09


All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


⚠️ No se encontró ningún checkpoint, cargando pesos preentrenados.
i: 0
25/25 [==============================] - 14s 559ms/step
LOSS: [11.377718, 11.428146, 11.685179, 11.428357, 11.494776, 11.608957, 11.681656, 11.664465, 11.455809, 11.701069, 11.655604, 11.589422, 11.501535, 11.425684, 11.780828, 11.438796, 11.656221, 11.471649, 11.444547, 11.399634, 11.420461, 11.687363, 11.483938, 11.585317, 11.515207]
Secrets #1, vocab=['Abel', 'Adam', 'Carl', 'Dean', 'Gill', 'Grey', 'Hunt', 'Leon', 'Lynn', 'Reed'], pattern={}{}{}{}{}
  Interpolation: repetition=1, avg_exposure=1.40±1.38; repetition=10, avg_exposure=1.05±1.22; repetition=100, avg_exposure=0.99±1.25
  Extrapolation: repetition=1, avg_exposure=1.79±1.94; repetition=10, avg_exposure=1.33±1.69; repetition=100, avg_exposure=1.57±3.09


We can see the following:
1. Comparing #1 vs. #2, having the prefix (#2) for the random sequences might help the model memorize.
2. Comparing #1 vs. #3, having the pattern (#3) did not help the memorization significantly. This is different from the result in the [paper](https://arxiv.org/abs/1802.08232). The effect might be model / data specific.
3. Comparing #1 vs. #4, longer secrets (#4) show higher exposure.

You can also try some other secret configurations to see if there is any difference.